In [2]:
import pandas as pd

In [29]:
def convert_transactions(transactions: list[dict]) -> pd.DataFrame: 
    """
    Converts list of transaction dictionaries into sorted pandas dataframe
    """
    
    # Convert transaction dictionaries into a dataframe
    df = pd.DataFrame(transactions)
    
    # Convert timestamp column to datetime
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    # Set timestamp as index and sort chronologically 
    df = df.set_index('timestamp').sort_index()
    
    return df
    
    

In [30]:
# Test generated by Claude

test_records = [
    {"timestamp": "2026-10-15 08:30:00", "register_id": "POS_01", "item": "Latte",      "price": 4.50,   "currency": "EUR", "status": "SUCCESS"},
    {"timestamp": "2026-10-15 07:12:30", "register_id": "POS_02", "item": "Espresso",   "price": 3.00,   "currency": "EUR", "status": "SUCCESS"},
    {"timestamp": "2026-10-15 07:45:10", "register_id": "POS_01", "item": "Muffin",     "price": 3.50,   "currency": "USD", "status": "REFUND"},
    {"timestamp": "2026-10-15 08:05:00", "register_id": "POS_03", "item": "Latte",      "price": 4.50,   "currency": "NOK", "status": "SUCCESS"},
    {"timestamp": "2026-10-15 07:59:59", "register_id": "POS_02", "item": "Sandwich",   "price": 999.00, "currency": "EUR", "status": "SUCCESS"},
    {"timestamp": "2026-10-15 09:10:20", "register_id": "POS_03", "item": "Cappuccino", "price": 4.20,   "currency": "EUR", "status": "CANCELLED"},
    {"timestamp": "2026-10-15 07:20:00", "register_id": "POS_01", "item": "Latte",      "price": 4.60,   "currency": "EUR", "status": "REFUND"},
    {"timestamp": "2026-10-15 09:45:00", "register_id": "POS_02", "item": "Latte",      "price": 4.30,   "currency": "USD", "status": "SUCCESS"},
]

df = convert_transactions(test_records)
print(df)
print(df.dtypes)

assert isinstance(df.index, pd.DatetimeIndex)        # index is datetime
assert df.index.is_monotonic_increasing              # sorted chronologically
assert "timestamp" not in df.columns                 # moved to the index
assert len(df) == len(test_records)                  # no rows lost
assert df.index[0] == pd.Timestamp("2026-10-15 07:12:30")
assert df.index[-1] == pd.Timestamp("2026-10-15 09:45:00")

                    register_id        item  price currency     status
timestamp                                                             
2026-10-15 07:12:30      POS_02    Espresso    3.0      EUR    SUCCESS
2026-10-15 07:20:00      POS_01       Latte    4.6      EUR     REFUND
2026-10-15 07:45:10      POS_01      Muffin    3.5      USD     REFUND
2026-10-15 07:59:59      POS_02    Sandwich  999.0      EUR    SUCCESS
2026-10-15 08:05:00      POS_03       Latte    4.5      NOK    SUCCESS
2026-10-15 08:30:00      POS_01       Latte    4.5      EUR    SUCCESS
2026-10-15 09:10:20      POS_03  Cappuccino    4.2      EUR  CANCELLED
2026-10-15 09:45:00      POS_02       Latte    4.3      USD    SUCCESS
register_id        str
item               str
price          float64
currency           str
status             str
dtype: object


In [ ]:
def group_items(transactions: pd.DataFrame) -> pd.DataFrame:
    """
    Takes a dataframe of transactions, groups data by item and resamples
    transactions in 1h windows, calculating total price
    """
    
    return transactions.groupby('item')['price'].resample('1h').sum()

In [32]:
group_items(df)

item        timestamp          
Cappuccino  2026-10-15 09:00:00      4.2
Espresso    2026-10-15 07:00:00      3.0
Latte       2026-10-15 07:00:00      4.6
            2026-10-15 08:00:00      9.0
            2026-10-15 09:00:00      4.3
Muffin      2026-10-15 07:00:00      3.5
Sandwich    2026-10-15 07:00:00    999.0
Name: price, dtype: float64

In [44]:
def status_total_value(transactions: pd.DataFrame) -> pd.DataFrame:
    """
    Takes a dataframe of transactions, and creates a pivot table displaying
    total transaction revenue grouped by item across status categories
    """
    
    pivot = transactions.pivot_table(
        values="price", 
        index="item", 
        columns="status", 
        aggfunc="sum", 
        fill_value=0
        )
    
    pivot = pivot.reindex(columns=["SUCCESS", "REFUND", "CANCELLED"], fill_value=0)

    return pivot

In [43]:
status_total_value(df)

status,CANCELLED,REFUND,SUCCESS
item,,,
Cappuccino,4.2,0.0,0.0
Espresso,0.0,0.0,3.0
Latte,0.0,4.6,13.3
Muffin,0.0,3.5,0.0
Sandwich,0.0,0.0,999.0


In [45]:
def refund_ratio(transactions: pd.DataFrame):
    """
    Takes a dataframe of transactions and calculates the refund ratio per item
    using compute_refund_ratio()
    """
    # Get number of transactions per item and status
    counts = (
        transactions.groupby(["item", "status"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=["SUCCESS", "REFUND", "CANCELLED"], fill_value=0)
    )

    # Make numpy ndarrays of sales and refund count
    sales = counts["SUCCESS"].to_numpy()
    refunds = counts["REFUND"].to_numpy()

    # Return dataframe with sales count, refund count, and ratio per item
    return pd.DataFrame(
        {
            "sales_count": sales,
            "refund_count": refunds,
            "refund_ratio": compute_refund_ratio(sales, refunds),
        },
        index=counts.index,
    )